In [2]:
!pip install python-dotenv

Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [29]:
!uv pip install langchain
!uv pip install langchain-community
!uv pip install langchain-google-genai

Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Resolved 35 packages in 31ms                                         
Uninstalled 2 packages in 17ms
Installed 2 packages in 4ms63                               
 - langchain-core==1.2.28
 + langchain-core==0.3.63
 - langsmith==0.7.29
 + langsmith==0.2.11
Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Checked 1 package in 10ms
Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Resolved 38 packages in 14ms                                         
Uninstalled 2 packages in 14ms
Installed 2 packages in 4ms28                               
 - langchain-core==0.3.63
 + langchain-core==1.2.28
 - langsmith==0.2.11
 + langsmith==0.7.29


In [30]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
HF_TOKEN = os.environ.get("HF_TOKEN")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

In [32]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser, StrOutputParser

from pydantic import BaseModel,Field

In [52]:
from langchain_google_genai import ChatGoogleGenerativeAI
# llm_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")#"gemini-2.5-flash")
llm_model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

In [34]:
prompt_txt = """{query}"""

prompt = ChatPromptTemplate.from_template(prompt_txt)

llm_chain = (
    prompt
    |
    llm_model
)


In [35]:
response = llm_chain.invoke({"query":"first 4 colours of rainbow"})
response.content

'The first four colors of the rainbow, in order, are:\n\n1.  **Red**\n2.  **Orange**\n3.  **Yellow**\n4.  **Green**'

In [36]:
response = llm_chain.invoke({"query":"and other three"})
response.content

'"And other three" is a very short phrase, so its meaning depends heavily on the context in which it\'s used. Here are a few possibilities and how you might interpret them:\n\n**Common Interpretations:**\n\n* **Continuing a List:** This is the most frequent use. It implies that a list of items has already been mentioned, and "and other three" means there are three more items that belong to that same category.\n    * **Example:** "We found two apples, a banana, **and other three** oranges." (Meaning: two apples, one banana, and three more oranges.)\n\n* **Indicating a Group of Three:** It can simply be a way of saying "three more" or "a group of three" without necessarily having a specific list preceding it.\n    * **Example:** "The first experiment yielded one result. **And other three** were observed in the second trial." (Meaning: three additional results were observed.)\n\n* **Referring to a Previously Mentioned Group:** If a group of items was just discussed, "and other three" coul

In [46]:

from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableWithMessageHistory

In [54]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnablePassthrough

# Load environment variables (Make sure GOOGLE_API_KEY is set)
load_dotenv()

# 1. Initialize the Model
model = llm_model

# 2. Define the Prompt 
# The 'history' key must match the MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 3. Create the basic Chain
chain = prompt | model

# 4. Define a dictionary to store chat histories
# In production, this would be a database (Redis, Postgres, etc.)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 5. Wrap the Chain with RunnableWithMessageHistory
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 6. Usage
# You must provide a session_id in the config
config = {"configurable": {"session_id": "user_123"}}

# First interaction
response1 = with_message_history.invoke(
    {"input": "Hi! My name is Shudhanshu."},
    config=config
)
print(f"AI: {response1.content}")

# Second interaction (Memory test)
response2 = with_message_history.invoke(
    {"input": "What is my name?"},
    config=config
)
print(f"AI: {response2.content}")

response3 = with_message_history.invoke(
    {"input": "what is 3 main colours in rainbow"},
    config=config
)
print(f"AI:{response3.content}")

response4 = with_message_history.invoke(
    {"input": "What is my name?"},
    config=config
)
print(f"AI: {response4.content}")


AI: [{'type': 'text', 'text': "Hello Shudhanshu! It's nice to meet you. How can I help you today?", 'extras': {'signature': 'ErQECrEEAb4+9vsxDe1BCGNNm1ex988lXlgAOGbq7YHSQpSmW+xtqAnDK7GWf5TrhD1XVfEZEUO1gHdeZ0n+JkddRmQC7CGQMM/yqTzY34/v59FwILP4fp2NHb7j85aRS9+n3cf47K3EIBIv9yuowMJjEfkHJ7JdIsDx9ubQrKS1H9jDq6aFrEX1S1sD+9MsKP8k7hMd4WRI/Djj1pb3VOL2DPmTTrd8rhfhplssV9eFpEnKT5jOxVKzJAN3Jc+URS/2/abcklu42bvNucQFd2AvugG/lj4PDbmI3dPPBLN4Ae01g25MsCCW18HTi9PDDUiAadXajradOg85gFrh+YXSbWyjEjVSNRSXypS1ETx8Ow5bJd8R/rqmqVPvxfUId7bRsuWArqactyu9oR0FkO4TSXuCvN3YnZMJ6XVUWvTbcvxKYxKiv5KqDN15MoPWdvSiMU1lvqcaUep/okjs+Se22oxWxfTlf8f37c9WpvGa5lRCreyOU0F49CnOBSj4pmg7EaYWALHZ6HfYxlIYi6VQMmL7Lw248uyzIUYPA8Dumt+3r3alIN3em5Kce/MCzyKU1hz96u1/gV9hyqHWM4y33AkJxmK3Iltmf1JHR0tVfFvJ7ww8cYzPcXCNZIxsRaR8kusjOx06HgjU3VOwQV8YhJykuMBhz0SxdDozdGQcRmK6BtBMIgd6bP4aQd1ytAxBHbbccPdB9WEGDK3riKaRh8EzmD527sP/OvQre9h3mC5cbGBI'}}]
AI: [{'type': 'text', 'text': 'Your name is Shudhanshu! How can I help you today?', 'extras': {'signature': 'EpwCC

In [55]:
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    # Optional: Logic to keep only the last 10 messages (5 turns)
    all_messages = store[session_id].messages
    if len(all_messages) > 10:
        store[session_id].clear()
        store[session_id].add_messages(all_messages[-10:])
        
    return store[session_id]

Using Vector db


In [56]:
!uv pip install langchain-chroma


Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Resolved 85 packages in 1.20s                                        
Prepared 30 packages in 13.10s                                           
Installed 30 packages in 78ms                               
 + bcrypt==5.0.0
 + build==1.4.2
 + chromadb==1.5.7
 + durationpy==0.10
 + flatbuffers==25.12.19
 + httptools==0.7.1
 + importlib-metadata==8.7.1
 + importlib-resources==6.5.2
 + kubernetes==35.0.0
 + langchain-chroma==1.1.0
 + mmh3==5.2.1
 + mpmath==1.3.0
 + oauthlib==3.3.1
 + onnxruntime==1.24.4
 + opentelemetry-api==1.41.0
 + opentelemetry-exporter-otlp-proto-common==1.41.0
 + opentelemetry-exporter-otlp-proto-grpc==1.41.0
 + opentelemetry-proto==1.41.0
 + opentelemetry-sdk==1.41.0
 + opentelemetry-semantic-conventions==0.62b0
 + overrides==7.7.0
 + pybase64==1.4.3
 + pypika==0.51.1
 + pyproject-hooks==1.2.0
 + requests-oauthlib==2.0.0
 + sympy==1.14.0
 + uvicorn==0.44.0
 + uvloop==0.22.1
 + wa

In [58]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

from langchain_chroma import Chroma

In [59]:
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')
llm_model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

chroma_db = Chroma(collection_name='history_db', embedding_function=embeddings)


In [61]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


In [65]:

from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


# 1. Setup Models & Vector Store
embeddings = GoogleGenerativeAIEmbeddings(model="embedding-001")
vectorstore = chroma_db

# Pre-fill vector store with some "Long Term Memory" facts
vectorstore.add_texts([
    "The client's favorite color is midnight blue.",
    "The project deadline is December 15th, 2025.",
    "The budget for the GenAI project is $50,000."
])
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 2. Define the Prompt
# It handles context (from VectorStore) and history (from ChatMessageHistory)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the following context to help answer: \n\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

model = llm_model

# 3. Helper to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Create the Chain
# This chain pulls from the retriever based on the user's input
context_chain = (lambda x: x["input"]) | retriever | format_docs

# The final chain construction
chain = (
    RunnablePassthrough.assign(context=context_chain)
    | prompt
    | model
)

# 5. Add Session History Management
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

full_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 6. Usage
config = {"configurable": {"session_id": "user_456"}}

# This will trigger the retriever for "budget" and check chat history
response = full_chain.invoke(
    {"input": "What is the budget for my project?"},
    config=config
)

print(response.content)

The budget for your project (GenAI project) is $50,000.
